## helpers/expand_benchmark_params

Runtime parameter-expansion notebook invoked as the first task in the
orchestrator (`0 - Job Setup` Job 3).

**Responsibilities**
1. Read the job parameters (`scales`, `sizes`, `dbx_strategies`, `sf_strategies`,
   and the three concurrency knobs).
2. Validate every parameter with an explicit human-readable error explaining
   what's wrong and how to fix it — the job fails here, with a clear message,
   rather than deep inside a benchmark notebook.
3. Cross-check `dbx_benchmark_concurrency` / `sf_benchmark_concurrency`
   against the child jobs' `max_concurrent_runs` via the Databricks SDK
   (protects against manual edits to the child jobs).
4. Slugify strategies and build Cartesian products; emit as task values.

**Emitted task values** (all JSON strings):
- `scales_json` — every requested scale (consumed by the test for_each tasks).
- `scales_to_generate_json` — subset of `scales_json` whose data is **not yet
  on the volume**. The data-generation for_each consumes this so existing
  datasets are never regenerated; if every scale already exists, the list is
  `[]` and the for_each runs zero iterations.
- `dbx_combos_json` — list of `{scale, size, strategy_raw, strategy_slug}`
- `sf_combos_json` — list of `{scale, size, strategy_raw, strategy_slug}`


In [ ]:
# --- Widgets (populated from {{job.parameters.*}}) ---
dbutils.widgets.removeAll()
dbutils.widgets.text("scales", "100", "Scales (comma-separated, GB)")
dbutils.widgets.text("sizes", "M", "Warehouse sizes (comma-separated)")
dbutils.widgets.text("dbx_strategies", "liquid_clustering", "Databricks strategies (comma-separated)")
dbutils.widgets.text("sf_strategies", "native_gen1", "Snowflake strategies (comma-separated)")
dbutils.widgets.text("datagen_concurrency", "2", "Data-generation for_each concurrency")
dbutils.widgets.text("dbx_benchmark_concurrency", "3", "Databricks benchmark for_each concurrency")
dbutils.widgets.text("sf_benchmark_concurrency", "3", "Snowflake benchmark for_each concurrency")
dbutils.widgets.text("dbx_child_job_name", "TPC-DS Databricks", "Databricks child job name")
dbutils.widgets.text("sf_child_job_name", "TPC-DS Snowflake", "Snowflake child job name")
dbutils.widgets.text("dbx_catalog", "", "Databricks Catalog (for volume scan)")
dbutils.widgets.text("dbx_schema",  "", "Databricks Schema (for volume scan)")
dbutils.widgets.text("dbx_volume",  "", "Volume name (for volume scan)")

### Parameter validation

Every parameter gets a targeted check. If anything is invalid the cell raises
`ValueError` with a bulleted list of every issue — the job will fail here
before attempting any benchmark.


In [ ]:
import re

VALID_SIZES = ["2XS", "XS", "S", "M", "L", "XL", "2XL", "3XL", "4XL"]
VALID_DBX_STRATEGIES = ["liquid_clustering", "liquid clustering", "iceberg", "iceberg_liquid_clustering"]
VALID_SF_STRATEGIES = ["native_gen1", "native_gen2", "iceberg_gen1", "iceberg_gen2", "native_gen2_adaptive", "iceberg_gen2_adaptive", "sf_external_iceberg"]
_SF_STRATEGY_RE = re.compile(r"^(native|iceberg)_gen[12](_adaptive)?$|^sf_external_iceberg$")

_errors = []


def _fail(msg):
    _errors.append(msg)


def _parse_scales(raw):
    if raw is None or not raw.strip():
        _fail(
            "`scales` is currently empty and it is invalid. "
            "It should be a comma-separated list of integer scale factors (GB) "
            "between 1 and 100000, such as '10,100,1000'."
        )
        return []
    items = [s.strip() for s in raw.split(",") if s.strip()]
    if not items:
        _fail(
            f"`scales` is currently set to {raw!r} with no usable values. "
            "It should be a comma-separated list of integer scale factors (GB) "
            "between 1 and 100000, such as '10,100,1000'."
        )
        return []
    parsed = []
    for it in items:
        try:
            v = int(it)
        except ValueError:
            _fail(
                f"`scales` contains {it!r} which is not an integer. "
                "It should be a comma-separated list of integer scale factors (GB) "
                "between 1 and 100000, such as '10,100,1000' (not JSON arrays like '[10,100]')."
            )
            continue
        if not (1 <= v <= 100000):
            _fail(
                f"`scales` contains {v}, outside the valid range [1, 100000]. "
                "It should be a comma-separated list of integer scale factors (GB), "
                "such as '10,100,1000'."
            )
            continue
        parsed.append(str(v))
    # De-duplicate while preserving order
    seen = set(); out = []
    for v in parsed:
        if v not in seen:
            seen.add(v); out.append(v)
    return out


def _parse_sizes(raw):
    if raw is None or not raw.strip():
        _fail(
            "`sizes` is currently empty and it is invalid. "
            f"It should be a comma-separated list of warehouse sizes from {VALID_SIZES}, "
            "such as 'S,M,L,XL'."
        )
        return []
    items = [s.strip().upper() for s in raw.split(",") if s.strip()]
    parsed = []
    for it in items:
        if it not in VALID_SIZES:
            _fail(
                f"`sizes` contains {it!r} which is not a valid warehouse size. "
                f"Valid values are {VALID_SIZES}. Example: 'S,M,L,XL'."
            )
            continue
        parsed.append(it)
    # dedup
    seen = set(); out = []
    for v in parsed:
        if v not in seen: seen.add(v); out.append(v)
    return out


def _parse_dbx_strategies(raw):
    if raw is None or not raw.strip():
        return []  # empty = skip the Databricks branch (validated below)
    items = [s.strip().lower() for s in raw.split(",") if s.strip()]
    parsed = []
    for it in items:
        if it not in VALID_DBX_STRATEGIES:
            _fail(
                f"`dbx_strategies` contains {it!r} which is not a valid Databricks strategy. "
                f"Valid values: {VALID_DBX_STRATEGIES}. "
                "Example: 'liquid_clustering' or 'liquid_clustering,iceberg,iceberg_liquid_clustering'."
            )
            continue
        parsed.append(it)
    # dedup
    seen = set(); out = []
    for v in parsed:
        if v not in seen: seen.add(v); out.append(v)
    return out


def _parse_sf_strategies(raw):
    if raw is None or not raw.strip():
        return []  # empty = skip the Snowflake branch (validated below)
    items = [s.strip().lower() for s in raw.split(",") if s.strip()]
    parsed = []
    for it in items:
        if not _SF_STRATEGY_RE.match(it):
            _fail(
                f"`sf_strategies` contains {it!r} which is not a valid Snowflake strategy. "
                "Expected pattern '{native|iceberg}_gen{1|2}'. "
                f"Valid values: {VALID_SF_STRATEGIES}."
            )
            continue
        parsed.append(it)
    # dedup
    seen = set(); out = []
    for v in parsed:
        if v not in seen: seen.add(v); out.append(v)
    return out


def _parse_int(name, raw, min_v, max_v, example):
    if raw is None or raw == "":
        _fail(f"`{name}` is currently empty. It should be an integer in [{min_v}, {max_v}], such as {example}.")
        return None
    try:
        v = int(raw)
    except ValueError:
        _fail(
            f"`{name}` is currently {raw!r} which is not an integer. "
            f"It should be an integer in [{min_v}, {max_v}], such as {example}."
        )
        return None
    if not (min_v <= v <= max_v):
        _fail(
            f"`{name}` is currently {v}, outside the valid range [{min_v}, {max_v}]. "
            f"Example: {example}."
        )
        return None
    return v


# --- Run parse+validate ---
scales = _parse_scales(dbutils.widgets.get("scales"))
sizes = _parse_sizes(dbutils.widgets.get("sizes"))
dbx_strategies = _parse_dbx_strategies(dbutils.widgets.get("dbx_strategies"))
sf_strategies = _parse_sf_strategies(dbutils.widgets.get("sf_strategies"))

# At least one platform must have strategies (empty list = skip that platform).
if not dbx_strategies and not sf_strategies:
    _fail("Both `dbx_strategies` and `sf_strategies` are empty. Provide at least one platform\u2019s strategies "
          "(e.g. for a Snowflake-only run set `sf_strategies` and leave `dbx_strategies` empty).")

datagen_concurrency = _parse_int("datagen_concurrency", dbutils.widgets.get("datagen_concurrency"), 1, 10, "2")
dbx_benchmark_concurrency = _parse_int("dbx_benchmark_concurrency", dbutils.widgets.get("dbx_benchmark_concurrency"), 1, 100, "3")
sf_benchmark_concurrency = _parse_int("sf_benchmark_concurrency", dbutils.widgets.get("sf_benchmark_concurrency"), 1, 100, "3")
max_clusters = _parse_int("max_clusters", dbutils.widgets.get("max_clusters"), 0, 100, "1")
# Snowflake adaptive warehouses reject QUERY_THROUGHPUT_MULTIPLIER=1; require 0 or >=2.
if max_clusters == 1 and any(st.endswith("_adaptive") for st in sf_strategies):
    _fail("`max_clusters` is 1, which Snowflake adaptive warehouses reject. "
          "For adaptive strategies (*_gen2_adaptive) set `max_clusters` to 0 (unlimited) or an integer >= 2.")

# --- Cross-check concurrency against child jobs' max_concurrent_runs via SDK ---
dbx_child_job_name = dbutils.widgets.get("dbx_child_job_name")
sf_child_job_name = dbutils.widgets.get("sf_child_job_name")

try:
    from databricks.sdk import WorkspaceClient
    _w = WorkspaceClient()

    def _check_child(job_name, widget_name, widget_value):
        if widget_value is None:
            return
        matches = [j for j in _w.jobs.list(name=job_name)]
        if not matches:
            _fail(
                f"Child job {job_name!r} not found in this workspace. "
                "Run `0 - Job Setup` first to create the child jobs, or update "
                f"the `{'dbx_child_job_name' if 'dbx' in widget_name else 'sf_child_job_name'}` parameter."
            )
            return
        for j in matches:
            max_conc = (j.settings.max_concurrent_runs if j.settings else None) or 1
            if widget_value > max_conc:
                _fail(
                    f"`{widget_name}` is currently {widget_value}, but the child job "
                    f"{job_name!r} (job_id={j.job_id}) has max_concurrent_runs={max_conc}. "
                    f"With the queued for_each task, iterations above {max_conc} will be "
                    "SKIPPED with MAXIMUM_CONCURRENT_RUNS_REACHED. "
                    f"Fix: either lower `{widget_name}` to <= {max_conc}, "
                    "or raise the child job's `max_concurrent_runs` "
                    "(in `0 - Job Setup` cell 9) and rerun the notebook."
                )

    _check_child(dbx_child_job_name, "dbx_benchmark_concurrency", dbx_benchmark_concurrency)
    _check_child(sf_child_job_name,  "sf_benchmark_concurrency",  sf_benchmark_concurrency)
except ImportError:
    print("WARNING: databricks-sdk not available on this cluster; skipping child-job concurrency check.")
except Exception as e:
    print(f"WARNING: concurrency check via SDK failed ({type(e).__name__}: {e}); "
          "parameter ranges were validated, but the check against the child jobs' "
          "max_concurrent_runs did not run.")


if _errors:
    header = f"Parameter validation failed ({len(_errors)} error(s)):"
    body = "\n\n".join(f"  {i+1}. {e}" for i, e in enumerate(_errors))
    raise ValueError(f"{header}\n\n{body}")

print("All parameters validated:")
print(f"  scales                    = {scales}")
print(f"  sizes                     = {sizes}")
print(f"  dbx_strategies            = {dbx_strategies}")
print(f"  sf_strategies             = {sf_strategies}")
print(f"  datagen_concurrency       = {datagen_concurrency}")
print(f"  dbx_benchmark_concurrency = {dbx_benchmark_concurrency}")
print(f"  sf_benchmark_concurrency  = {sf_benchmark_concurrency}")

### Expand combos and emit task values

In [ ]:
import json


def strategy_slug(raw: str) -> str:
    return (re.sub(r"\W+", "_", (raw or "").strip()).strip("_") or "DEFAULT").upper()


dbx_combos = [
    {"scale": sc, "size": sz, "strategy_raw": st, "strategy_slug": strategy_slug(st)}
    for sc in scales for sz in sizes for st in dbx_strategies
]
sf_combos = [
    {"scale": sc, "size": sz, "strategy_raw": st, "strategy_slug": strategy_slug(st)}
    for sc in scales for sz in sizes for st in sf_strategies
]

# --- Decide which scales still need data generation ---
# A scale's data is considered "present" if at least 20 child directories
# under /Volumes/{catalog}/{schema}/{volume}/{scale}/ each contain at least
# one .dat file. Matches the safety-net check in `1 - Raw Data Generator`.

dbx_catalog_w = dbutils.widgets.get("dbx_catalog").strip()
dbx_schema_w  = dbutils.widgets.get("dbx_schema").strip()
dbx_volume_w  = dbutils.widgets.get("dbx_volume").strip()

if not (dbx_catalog_w and dbx_schema_w and dbx_volume_w):
    raise ValueError(
        "Missing one of dbx_catalog / dbx_schema / dbx_volume widgets. "
        "These are required so this notebook can scan the raw-data volume "
        "and skip generation for scales that already exist."
    )

_volume_root = f"/Volumes/{dbx_catalog_w}/{dbx_schema_w}/{dbx_volume_w}"

def _scale_data_exists(scale: str) -> bool:
    scale_path = f"{_volume_root}/{scale}"
    try:
        children = [c for c in dbutils.fs.ls(scale_path) if c.isDir()]
    except Exception:
        return False
    if len(children) < 20:
        return False
    populated = 0
    for c in children:
        try:
            if any(f.name.endswith(".dat") for f in dbutils.fs.ls(c.path)):
                populated += 1
        except Exception:
            continue
        if populated >= 20:
            return True
    return populated >= 20

scales_to_generate = [s for s in scales if not _scale_data_exists(s)]
scales_already_present = [s for s in scales if s not in scales_to_generate]

print(f"Volume root          : {_volume_root}")
print(f"Scales requested     : {scales}")
print(f"Scales already on volume: {scales_already_present}  (will skip data gen)")
print(f"Scales to generate   : {scales_to_generate}")

scales_json              = json.dumps(scales)
scales_to_generate_json  = json.dumps(scales_to_generate)
dbx_combos_json          = json.dumps(dbx_combos)
sf_combos_json           = json.dumps(sf_combos)

print(f"dbx_combos ({len(dbx_combos)}):")
for c in dbx_combos[:6]:
    print(f"  {c}")
if len(dbx_combos) > 6:
    print(f"  ... and {len(dbx_combos) - 6} more")
print()
print(f"sf_combos ({len(sf_combos)}):")
for c in sf_combos[:6]:
    print(f"  {c}")
if len(sf_combos) > 6:
    print(f"  ... and {len(sf_combos) - 6} more")

dbutils.jobs.taskValues.set("scales_json", scales_json)
dbutils.jobs.taskValues.set("scales_to_generate_json", scales_to_generate_json)
dbutils.jobs.taskValues.set("dbx_combos_json", dbx_combos_json)
dbutils.jobs.taskValues.set("sf_combos_json", sf_combos_json)

print()
print("Task values emitted: scales_json, scales_to_generate_json, dbx_combos_json, sf_combos_json")